# Saving graphs for neural-lam

From the first tagged version of [neural-lam](https://github.com/mllam/neural-lam) ([v0.1.0](https://github.com/mllam/neural-lam/tree/v0.1.0) and currently, as of `v0.4.0`) has a very specific format that the saved graphs are expected to be in based on pickled pytorch tensors. This notebook describes how to save graphs in that format and what the format is.

In [ ]:
import weather_model_graphs as wmg
import numpy as np
import matplotlib.pyplot as plt
import torch
import networkx as nx

We will start by creating some a graph from some randomly generated coordinates:

In [ ]:
def create_fake_irregular_coords(num_grid_points=100):
    """
    Create fake grid points on random coordinates
    """
    rng = np.random.default_rng(seed=42)  # Fixed seed
    # All coordinates in [0,1]^2
    return rng.random((num_grid_points, 2))

In [ ]:
coords = create_fake_irregular_coords(num_grid_points=12)

fig, ax = plt.subplots()
ax.scatter(coords[:, 0], coords[:, 1])
# add labels
for i, (x, y) in enumerate(coords):
    ax.text(x, y, str(i), fontsize=12, ha="right")
ax.set_aspect("equal")
ax.set_xlim(0, 1), ax.set_ylim(0, 1)

In [ ]:
graph = wmg.create.archetype.create_keisler_graph(coords=coords, mesh_node_distance=0.2)

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect("equal")
wmg.visualise.nx_draw_with_pos_and_attr(
    graph=graph, ax=ax, node_color_attr="type", edge_color_attr="component"
)

`neural-lam` expects the _subgraphs_ that are individually used for subsequent message passing operations to be saved separately. For the Keisler graph, this means that we need to split the graph into subgraphs that do the mesh-to-grid (`m2g`), mesh-to-mesh (`m2m`), grid-to-mesh (`g2m`), and grid-to-grid (`g2g`) operations. In `weather-model-graphs` edges have a `component` attribute that indicates which of these operations they belong to, so we can use this to split the graph.

In [ ]:
subgraphs = wmg.split_graph_by_edge_attribute(graph=graph, attr="component")
subgraphs

In [ ]:
outputdir = "keisler_graph_for_neural_lam"
for subgraph_name, subgraph_dict in subgraphs.items():
    wmg.save.to_pyg(subgraph_dict, output_directory=outputdir, name=subgraph_name)

! tree {outputdir}

In [ ]:
def load_nl_subgraph_format(base_path, graph_name):
    adj_list = torch.load(f"{base_path}/{graph_name}_edge_index.pt")
    edge_features = torch.load(f"{base_path}/{graph_name}_features.pt")
    node_features = torch.load(f"{base_path}/{graph_name}_node_features.pt")
    return {
        "edge_adj_list": adj_list,
        "edge_features": edge_features,
        "node_features": node_features,
    }


nl_g2m_subgraph_dict = load_nl_subgraph_format("keisler_graph_for_neural_lam", "g2m")
nl_g2m_subgraph_dict

Let's start by looking shapes of the edge adjacency list, edge features and node features for the three subgraphs:

In [ ]:
def print_shapes_of_nl_subgraphs(basedir, graph_components=["g2m", "m2m", "m2g"]):
    for component in graph_components:
        subgraph_dict = load_nl_subgraph_format(basedir, component)
        print(f"Component: {component}")
        print(f"  Edge adjacency list shape:   {subgraph_dict['edge_adj_list'].shape}")
        print(f"  Edge features shape:         {subgraph_dict['edge_features'].shape}")
        print(
            f"  Node features shape:         {[nf.shape for nf in subgraph_dict['node_features']]}"
        )
        print()


print_shapes_of_nl_subgraphs("keisler_graph_for_neural_lam")

Next we will will construct `networkx.DiGraph` objects for each of these subgraphs so we can visualise them and check that they look correct.

In [ ]:
def graph_from_nl_subgraph_format(nl_subgraph_dict):
    g = nx.DiGraph()
    edge_adj_list = nl_subgraph_dict["edge_adj_list"].numpy()
    edge_features = nl_subgraph_dict["edge_features"].numpy()
    node_features = [nf.numpy() for nf in nl_subgraph_dict["node_features"]]

    n_edges = edge_adj_list.shape[1]
    assert (
        edge_features.shape[0] == n_edges
    ), "Number of edges in adjacency list and features do not match"
    n_nodes = node_features[0].shape[0]
    assert (
        len(np.unique(edge_adj_list)) == n_nodes
    ), "Number of unique nodes in adjacency list exceeds number of node features"

    # add nodes
    for i in range(n_nodes):
        nf = node_features[0][i]
        # assume node features are [pos[0], pos[1]]
        node_pos = (nf[0], nf[1])
        g.add_node(i, pos=node_pos)

    # Add edges
    for i in range(edge_adj_list.shape[1]):
        src = int(edge_adj_list[0, i])
        dst = int(edge_adj_list[1, i])
        # assume edges features are [len, vec[0], vec[1]]
        e_features = edge_features[i]
        edge_features_named = {
            "len": float(e_features[0]),
            "vec": (float(e_features[1]), float(e_features[2])),
        }
        g.add_edge(src, dst, **edge_features_named)

    return g


nl_g2m_subgraph = graph_from_nl_subgraph_format(nl_g2m_subgraph_dict)
nl_g2m_subgraph

In [ ]:
kwargs = dict(edge_color_attr="len")
fig, axes = plt.subplots(ncols=2, figsize=(12, 6))
[ax.set_aspect("equal") for ax in axes]
# original graph
ax = axes[0]
wmg.visualise.nx_draw_with_pos_and_attr(graph=subgraphs["g2m"], ax=ax, **kwargs)
# loaded graph
ax = axes[1]
wmg.visualise.nx_draw_with_pos_and_attr(graph=nl_g2m_subgraph, ax=ax, **kwargs)

## Assumptions in neural-lam pytorch subgraph disk format

It important to note how the *label* of nodes in `networkx.DiGraph` created by `weather-model-graphs` is used since these labels define the node-index values. In `neural-lam`, the node indices are expected to be contiguous integers starting from zero. The node labels in the `networkx.DiGraph` objects created by `weather-model-graphs` are exactly these integer indices, and writing to adjancy lists and feature tensors is done using these integer indices.


We will start by creating a plot which shows how the three subgraphs connect the mesh and grid nodes together, directly from the `networkx.DiGraph` objects we created above.

In [ ]:
def plot_subgraph_edge_connections(subgraphs):

    fig, ax = plt.subplots(figsize=(10, 6))
    y_ticks = []
    y_ticklabels = []

    offset = -0.4
    colors = dict(
        g2m="C0",
        m2m="C1",
        m2g="C2",
    )
    component_names = sorted(list(subgraphs.keys()))
    for i, component in enumerate(component_names):
        color = colors[component]
        subgraph = subgraphs[component]
        edge_adj_list = np.array(subgraph.edges()).T
        ys = [i - offset, i + offset]
        ys_labels = [f"{component} src", f"{component} dst"]
        for src, dst in edge_adj_list.T:
            ax.plot([src, dst], ys, color=color)
        y_ticks += ys
        y_ticklabels += ys_labels

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_ticklabels)
    ax.set_ylabel("Subgraph component and node role")
    ax.set_xlabel("Node index")


plot_subgraph_edge_connections(subgraphs)

You can see from this plot that the node indices are ordered so that the mesh nodes come first, followed by the grid nodes. The edges in each subgraph connect the appropriate nodes together.

If we repeat this with the graphs saved into neural-lam pytorch format, we can see that there is a bug here currently in `weather-model-graphs` where the grid-to-grid subgraph is not being saved correctly. This will be fixed in a future release.

In [ ]:
# create a plot where along the x-axis is the grid-index and along y-axis is
# the subgraph name, with an entry along y for both source and target nodes


def plot_nl_subgraph_edge_connections(subgraph_dicts):

    fig, ax = plt.subplots(figsize=(10, 6))
    y_ticks = []
    y_ticklabels = []

    offset = -0.4
    colors = dict(
        g2m="C0",
        m2m="C1",
        m2g="C2",
    )
    component_names = sorted(list(subgraphs.keys()))
    for i, component in enumerate(component_names):
        color = colors[component]
        subgraph_dict = subgraph_dicts[component]
        edge_adj_list = subgraph_dict["edge_adj_list"].numpy()
        ys = [i - offset, i + offset]
        ys_labels = [f"{component} src", f"{component} dst"]
        for src, dst in edge_adj_list.T:
            ax.plot([src, dst], ys, color=color)
        y_ticks += ys
        y_ticklabels += ys_labels

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_ticklabels)
    ax.set_ylabel("Subgraph component and node role")
    ax.set_xlabel("Node index")


subgraph_dicts = {
    component: load_nl_subgraph_format("keisler_graph_for_neural_lam", component)
    for component in ["g2m", "m2m", "m2g"]
}
plot_nl_subgraph_edge_connections(subgraph_dicts)